In [1]:
%pip install -U openai python-dotenv

  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   -------------------- ------------------- 1.0/2.1 MB 3.1 MB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.1 MB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 3.4 MB/s  0:00:00
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)

   -------------------------- ------------- 4/6 [httpx2]
   --------------------------------- ------ 5/6 [openai]
   --------------------------------- ------ 5/6 [openai]
   --------------------------------- ------ 5/6 [openai]
   --------------------------------- ------ 5/6 [openai]
   --------------------------------- ------ 5/6 [openai]
   --------------------------------- ------ 5/6 [openai]
   --------------------------------- ------ 5/6 [openai]
   --------------------------------- ------ 5/6 [openai]
   ----

In [1]:
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import os

project_root = Path.cwd().parent
env_file = project_root / ".env"

load_dotenv(env_file)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY was not found in the .env file.")

client = OpenAI(api_key=api_key)
del api_key

print("OpenAI API key loaded successfully.")
print("OpenAI client created successfully.")

OpenAI API key loaded successfully.
OpenAI client created successfully.


In [2]:
import chromadb
from sentence_transformers import SentenceTransformer

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(embedding_model_name)

chroma_path = project_root / "data" / "chroma_db"
chroma_client = chromadb.PersistentClient(path=str(chroma_path))

collection = chroma_client.get_collection(
    name="novamart_support"
)

print("Embedding model loaded.")
print(f"Connected to ChromaDB with {collection.count()} chunks.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.
Connected to ChromaDB with 27 chunks.


In [3]:
def retrieve_chunks(question, number_of_results=3):
    question_embedding = embedding_model.encode(
        question,
        normalize_embeddings=True
    )

    results = collection.query(
        query_embeddings=[question_embedding.tolist()],
        n_results=number_of_results,
        include=["documents", "metadatas", "distances"]
    )

    retrieved_chunks = []

    for position in range(number_of_results):
        retrieved_chunks.append({
            "content": results["documents"][0][position],
            "source": results["metadatas"][0][position]["source"],
            "similarity": 1 - results["distances"][0][position]
        })

    return retrieved_chunks

print("Retrieval function is ready.")

Retrieval function is ready.


In [4]:
def answer_question(question, number_of_results=3):
    retrieved_chunks = retrieve_chunks(
        question,
        number_of_results=number_of_results
    )

    context_sections = []

    for chunk in retrieved_chunks:
        context_sections.append(
            f"Source: {chunk['source']}\n"
            f"Content:\n{chunk['content']}"
        )

    context = "\n\n---\n\n".join(context_sections)

    response = client.responses.create(
        model="gpt-6-luna",
        instructions=(
            "You are NovaMart Australia's customer support assistant. "
            "Answer using only the supplied policy context. "
            "Do not use outside knowledge or invent company policies. "
            "If the context does not contain enough information, say that "
            "you cannot answer from the available NovaMart policies. "
            "Give a concise, helpful answer and cite the supporting source "
            "filename in square brackets, for example [returns_refunds.md]."
        ),
        input=(
            f"Customer question:\n{question}\n\n"
            f"NovaMart policy context:\n{context}"
        ),
        max_output_tokens=300
    )

    return {
        "question": question,
        "answer": response.output_text,
        "retrieved_chunks": retrieved_chunks
    }

print("Complete RAG function is ready.")

Complete RAG function is ready.


In [5]:
result = answer_question(
    "My order arrived damaged. What should I do?"
)

print("Question:")
print(result["question"])

print("\nGenerated answer:")
print(result["answer"])

Question:
My order arrived damaged. What should I do?

Generated answer:
Contact NovaMart Support within 7 calendar days of delivery. Include your order number and clear photos of the damaged product and packaging. NovaMart will provide a prepaid return label, and you can choose a replacement, a repair where appropriate, or a full refund to your original payment method. [returns_refunds.md]


In [6]:
unsupported_result = answer_question(
    "Does NovaMart match a competitor's lower price?"
)

print("Question:")
print(unsupported_result["question"])

print("\nGenerated answer:")
print(unsupported_result["answer"])

Question:
Does NovaMart match a competitor's lower price?

Generated answer:
I can’t confirm whether NovaMart matches competitors’ lower prices from the available policies. [shipping_delivery.md]


In [7]:
print("Retrieved evidence for the unsupported question:")

for position, chunk in enumerate(
    unsupported_result["retrieved_chunks"],
    start=1
):
    print(f"\nResult {position}")
    print("Source:", chunk["source"])
    print(f"Similarity: {chunk['similarity']:.3f}")
    print("Section:", chunk["content"].splitlines()[2])

Retrieved evidence for the unsupported question:

Result 1
Source: shipping_delivery.md
Similarity: 0.483
Section: ## Standard delivery

Result 2
Source: account_privacy.md
Similarity: 0.454
Section: ## Personal information

Result 3
Source: product_warranty.md
Similarity: 0.434
Section: ## Australian Consumer Law


In [8]:
def answer_question(question, number_of_results=3):
    retrieved_chunks = retrieve_chunks(
        question,
        number_of_results=number_of_results
    )

    context_sections = []

    for chunk in retrieved_chunks:
        context_sections.append(
            f"Source: {chunk['source']}\n"
            f"Content:\n{chunk['content']}"
        )

    context = "\n\n---\n\n".join(context_sections)

    response = client.responses.create(
        model="gpt-6-luna",
        instructions=(
            "You are NovaMart Australia's customer support assistant. "
            "Answer using only the supplied policy context. "
            "Do not use outside knowledge or invent company policies. "
            "Cite only a source that directly supports your answer. "
            "If the context does not directly answer the customer's question, "
            "respond exactly with: "
            "'I cannot answer this from the available NovaMart policies.' "
            "Do not include a citation when giving this refusal. "
            "Otherwise, give a concise and helpful answer and cite the "
            "supporting source filename in square brackets, for example "
            "[returns_refunds.md]."
        ),
        input=(
            f"Customer question:\n{question}\n\n"
            f"NovaMart policy context:\n{context}"
        ),
        max_output_tokens=300
    )

    return {
        "question": question,
        "answer": response.output_text,
        "retrieved_chunks": retrieved_chunks
    }

print("RAG function updated with stronger citation rules.")

RAG function updated with stronger citation rules.


In [9]:
unsupported_result = answer_question(
    "Does NovaMart match a competitor's lower price?"
)

print("Question:")
print(unsupported_result["question"])

print("\nGenerated answer:")
print(unsupported_result["answer"])

Question:
Does NovaMart match a competitor's lower price?

Generated answer:
I cannot answer this from the available NovaMart policies.


In [10]:
supported_result = answer_question(
    "How long is the password reset link valid?"
)

print("Question:")
print(supported_result["question"])

print("\nGenerated answer:")
print(supported_result["answer"])

Question:
How long is the password reset link valid?

Generated answer:
The password reset link is valid for 30 minutes. [account_privacy.md]
